# Assignment 1 — Python, pandas, and Data That Fights Back


Graded on completion. You may work with others; each of you submits your own copy.

---

**Your name / NYU email: Haoyu Huangfu/hh3170@nyu.edu

**Worked with: Gemini

---

This is the warm-up. It exists because the rest of the term assumes you can get
data into Python and trust what comes out.

**Use AI for all of it.** Every line of code here is something Gemini or ChatGPT
will write for you in seconds, and you should let it. What the AI will *not* do
is notice when the answer is wrong — and this assignment is mostly a tour of
ways a notebook returns a confident, plausible, incorrect number.

Where a question asks you to **predict before running**, do that honestly. The
prediction is the point; getting it wrong costs you nothing.

---

# Part 1 — Two ways a notebook will lie to you

### Q1 — Cells run in the order *you* run them, not the order they appear

Run the next two cells in order. You should get 2,680.19.

In [5]:
deposit = 2000
rate    = 0.05
years   = 6

In [6]:
deposit * (1 + rate) ** years

2680.191281250001

Now, in the cell below, change the rate to **10%** — and do *not* re-run
anything above.

In [3]:
rate = 0.10

Go back and re-run **only** the `deposit * (1 + rate) ** years` cell.

> **(a)** What number do you get now, and why is it not the 5% answer?
>
> **(b)** Now scroll up and re-run the *first* cell, then the calculation cell
> again. What happens, and why?
>
> **(c)** A notebook has an execution *order* and a visual *order*, and they are
> not the same thing. Describe, in one or two sentences, how this could produce a
> number in your final report that you cannot reproduce the next morning.

This is the single most common way a Jupyter result turns out to be wrong, and
AI makes it worse rather than better — pasting a fix halfway up the notebook and
re-running one cell is exactly the move that causes it.

**(a)** 3543.122 bc rate be changed

**(b)** back to 2680.19 bc rate be set up back to 5%

**(c)** the variables in memory depend on the execution order. When you reopen the notebook the next day and run it top-to-bottom (the visual order), those variables will have different values, producing a different final result.

### Q2 — Code that runs, returns a plausible number, and is wrong

You put \$100 into a stock. It returns **+15%** in year one and **−14%** in
year two. The cell below claims to compute what you end up with.

**Do not fix it yet.** First: look at it, and write down what you think it will
print.

In [7]:
r1 = 0.15
r2 = -0.14

100 * 1 + r1 * 1 + r2      # <- what will this print?

100.01

> **(a)** Your prediction, before running:
>
> **(b)** Run it. What did it actually print? Is that number plausible as
> "\$100 after two years"? Would you have caught it if it had appeared in the
> middle of a table?
>
> **(c)** Fix it in the cell below, and state the correct final value.

In [8]:
# Your corrected calculation
100 * (1 + r1) * (1 + r2)

98.89999999999999

**(a)** prediction: 100

**(b)** It printed 100.01. Yes, it is very plausible bc a 15% gain followed by a 14% loss feels like it should be roughly break-even, making it hard to catch if it were buried in a table.

**(c)** correct value: 98.90

---

# Part 2 — Python worth knowing when the AI writes the code

### Q3 — Predict, then run

Below are six statements. `x = 2`, `y = 2`, `z = 4`.

```python
x > z                              # 1
x == y                             # 2
(x < y) and (x > y)                # 3
(x < y) or  (x > y)                # 4
(x <= y) and (x >= y)              # 5
True and ((x < z) or (x < y))      # 6
```

**Write down your six answers first**, as a list like `[True, False, ...]`.
Then run them and compare. Note any you got wrong — `and`/`or` precedence is a
real source of silently wrong filters later in the course.

**My predictions:** `[ , , , , , ]`

In [9]:
x, y, z = 2, 2, 4
# check your six predictions here
[False,True,False,False,True,False ]

[False, True, False, False, True, False]

### Q4 — Data does not arrive as numbers

You are handed a price as `"$6.50"`. Python sees a string, and `"$6.50" * 2`
gives you `"$6.50$6.50"` rather than 13.

Turn it into the float `6.5`. Then say in one sentence what would happen if a
column of a thousand prices had this problem and you took its mean.

In [11]:
price = "$6.50"
# your code here
price_float = float(price.replace('$', ''))
print(price_float)

6.5


**What happens to the mean of a column like this:**
Python cannot do math with text. If you try to find the average of a column of strings, the code will just crash with an error.

### Q5 — Why logs show up everywhere in finance

There is a trick worth knowing: for numbers close to 1, the percent change
$(x-y)/y$ is well approximated by the difference in logs, $\log x - \log y$.

Verify it with the numbers below — compute both and compare. Then try it again
with `x = 2.0, y = 1.0` and report how well the approximation holds.

One sentence: when is it safe to use, and when is it not?

In [12]:
import math

x, y = 1.05, 1.02
# your code here
pct_change_1 = (x - y) / y
log_diff_1 = math.log(x) - math.log(y)
print(f"For x={x}, y={y}:")
print(f"Percent change: {pct_change_1:.5f}")
print(f"Log difference: {log_diff_1:.5f}")

x2, y2 = 2.0, 1.0
pct_change_2 = (x2 - y2) / y2
log_diff_2 = math.log(x2) - math.log(y2)
print(f"\nFor x={x2}, y={y2}:")
print(f"Percent change: {pct_change_2:.5f}")
print(f"Log difference: {log_diff_2:.5f}")

For x=1.05, y=1.02:
Percent change: 0.02941
Log difference: 0.02899

For x=2.0, y=1.0:
Percent change: 1.00000
Log difference: 0.69315


**When it holds, and when it breaks:**
The log approximation is safe when numbers are very close to each other, but it breaks down and becomes highly inaccurate for large jumps.

---

# Part 3 — Real data, which does not want to help you

Everything from here uses one Excel file: 49 US industry portfolios from Ken
French, monthly, going back to 1926, plus a sheet with the market return and the
risk-free rate.

```
https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/Assignment1.xlsx
```

**Download it and open it in Excel before you write any code.** Ten seconds of
looking will save you an hour. This is a habit, not a suggestion.

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 4]

URL = ("https://raw.githubusercontent.com/amoreira2/UG54/"
       "refs/heads/main/assets/data/Assignment1.xlsx")
pd.ExcelFile(URL).sheet_names

['Market_proxy', '49_Industry_Portfolios']

### Q6 — Load it, and count the columns

Read the `49_Industry_Portfolios` sheet. The real column names are **not** in
the first row — there are several lines of description above them, so you will
need `skiprows`.

Then answer this before going further: **how many columns did you get?**

> **(a)** How many columns are there, and how many industries were you promised?
>
> **(b)** Print the column names. Some of them end in `.1` and `.2`. What has
> happened, and what would you have computed if you had just averaged everything?
>
> **(c)** Keep only the block you actually want — the value-weighted returns —
> and say how you know it is the right one.

In [14]:
raw = pd.read_excel(URL, sheet_name='49_Industry_Portfolios', skiprows=11)
print(f"Total columns: {raw.shape[1]}")
print(f"Columns: {list(raw.columns)}")

Total columns: 152
Columns: [192611, 6.75, 6.26, -99.99, 7.29, 4.55, 0, 1.82, -6.4, -0.54, 1.87, '-99.99.1', 4.33, 5.76, 5.2, '-99.99.2', 3.11, 2.2, 2.25, 3.86, '-99.99.3', 3.18, 5.08, -0.66, 6.61, 7.89, '-99.99.4', '-99.99.5', 8.46, -0.48, 0.06, 3.71, 1.63, '-99.99.6', 0.78, 3.28, '-99.99.7', 1.31, 4.25, '-99.99.8', 3.84, 1.6, 4.67, 6.52, '4.33.1', -2.97, 3.58, 2.21, 4.92, 4, 'Unnamed: 50', '192611.1', 102.34, 30.9, '-99.99.9', 8.62, 65.08, 17.62, 29.01, 9.31, 20.14, 19.03, '-99.99.10', 22.17, 28.29, 65.06, '-99.99.11', 6.84, 21.05, 8.04, 48.9, '-99.99.12', 29.6, 108.33, 54.81, 7.07, 43.22, '-99.99.13', '-99.99.14', 28.51, 41.9, 89.87, 83.92, 363.74, '-99.99.15', 12.17, 31.61, '-99.99.16', 21.91, 240.92, '-99.99.17', 32.8, 70.58, 0.74, 45.03, 10.33, 14.52, 31.26, 22.33, 9.56, 23.94, 'Unnamed: 101', 1930, 0.79, 0.47, '-99.99.18', 1.7, 1.3, 2.05, '0.79.1', 0.87, 0.33, 0.86, '-99.99.19', 0.16, 0.49, 0.41, 0.89, 2.19, 0.72, 0.42, 1.03, '-99.99.20', 0.58, 0.31, 0.7, 0.59, 1, '-99.99.21', '

**(a)** There are 152 columns, but we were only promised 49 industries.

**(b)** The `.1` and `.2` suffixes were added by pandas because there are duplicate column names. This happens because the Excel sheet contains multiple tables (e.g., value-weighted returns, equal-weighted returns) side-by-side or stacked. If we averaged everything without filtering, we would be averaging across completely different metrics.

**(c)** We know the first block is the value-weighted returns because the file documentation usually lists it first. We can extract it by keeping just the first 50 columns (the date column plus the 49 industries).

In [15]:
ind = raw.iloc[:, :50]        # date column + the 49 value-weighted industries only
print(ind.shape)

(1064, 50)


### Q7 — Missing values that are not missing

The header text says: *"All missing values are indicated by -99.99 or -999."*

Find them and replace them with proper `NaN`. Then report what fraction of the
table was missing.

> **⚠️** If you skip this, `-99.99` is a perfectly valid number and every mean,
> standard deviation and correlation you compute will quietly include it as a
> **−99.99% monthly return**. Nothing will error.

In [16]:
import numpy as np

# Replace sentinel missing values with proper NaN
ind = ind.replace([-99.99, -999], np.nan)

# Calculate the fraction of missing values
total_missing = ind.isna().sum().sum()
total_cells = ind.size
fraction_missing = total_missing / total_cells

print(f"Fraction missing: {fraction_missing:.2%}")

Fraction missing: 5.32%


### Q8 — Dates and units

Two more things stand between you and usable data.

**Dates.** The first column is an integer like `192607`. Convert it to a proper
month-end date and make it the index. Month-*end* matters: these are monthly
returns, and later in the course you will merge them against other series that
are stamped at month end. A one-day mismatch silently drops every row.

**Units.** Print the mean of one industry. Is it a monthly return of 0.9%, or
90%? Convert so that a 1% return is stored as `0.01`.

In [17]:
# Extract the first column for dates
date_col = ind.columns[0]
dates = pd.to_datetime(ind[date_col].dropna().astype(int).astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)

# Set the index and remove the original column
ind = ind.set_index(dates)
ind = ind.drop(columns=[date_col])
ind.index.name = 'Date'

# Convert percentages to decimals
ind = ind / 100

print(ind.index[:3])

# Safely attempt to print 'Agric' (handles potential missing headers from Q6)
try:
    print(f"mean monthly return, Agric: {ind['Agric'].mean():.5f}")
except KeyError:
    print("Column 'Agric' not found (headers might be missing). Mean of first industry:")
    print(f"{ind.iloc[:, 0].mean():.5f}")

DatetimeIndex(['1926-12-31', '1927-01-31', '1927-02-28'], dtype='datetime64[ns]', name='Date', freq=None)
Column 'Agric' not found (headers might be missing). Mean of first industry:
0.00959


### Q9 — Excess returns

Load the `Market_proxy` sheet the same way — it has `Mkt-RF` and `RF`. Watch the
units here too.

Then build excess returns two ways:

1. For **one** industry, `Agric`, subtract `RF` month by month. Print its mean.
2. For **all 49 at once**, in a single line, producing a DataFrame called `inde`.

> **🤖 Worth asking the AI:** *"subtract a Series from every column of a
> DataFrame, aligning on the index"* — and then check that the row count did not
> change. Getting the axis wrong here produces a table full of NaN, or worse, a
> table that looks fine and is transposed.

### Q10 — Dropping rows has a price

Not every industry exists in 1926. To get a rectangular table where every
industry covers the same months, drop the incomplete rows.

Report: how many months you had before, how many after, and **what date the
sample now starts**.

> **(a)** How many months did that cost you?
>
> **(b)** You have just thrown away 40 years of data to keep 49 columns. Name
> one thing you might have done instead, and say what it would have cost.

In [ ]:
# your code here


**(a)**

**(b)**

### Q11 — The two objects everything else is built from

Compute and display:

1. `ERe` — the vector of **annualized** mean excess returns, one per industry
2. `CovRe` — the **annualized** covariance matrix of excess returns

> **If you have not met a covariance matrix.** For 49 industries it is a 49×49
> table. The diagonal entry $(i,i)$ is industry $i$'s **variance** — its
> volatility squared. The off-diagonal entry $(i,j)$ is the **covariance**
> between $i$ and $j$: positive when the two tend to move in the same direction,
> negative when one tends to rise as the other falls, near zero when neither
> tells you much about the other.
>
> A **correlation** is that same covariance divided by the two volatilities,
> which rescales it to sit between −1 and +1. Same information, easier to read —
> which is why the question below asks you for correlations rather than raw
> covariances.
>
> `df.cov()` builds the whole table in one call. Nothing in the next few weeks
> needs it. It comes back in Capital Allocation in October, where it is the
> object that decides how much of each thing to hold.

Report the highest and lowest average-return industries. Then print the
correlation between three pairs of your choosing — pick one pair you expect to
move together and one you expect not to, and say whether the data agreed.

Remember: means annualize by ×12, variances by ×12, volatilities by ×√12.

In [ ]:
# your code here


**Pairs I picked, what I expected, what I found:**

### Q12 — Look at it

Pick **two** industries that you think behaved very differently in some
identifiable episode — a crisis, a boom, a technological shift.

1. Plot their monthly excess returns over that period.
2. Plot the **cumulative** return of \$1 invested in each over that period.
3. In three or four sentences, say what happened, using the actual magnitudes
   from your plot. What would \$1 have become in each?

> **📌** Cumulative returns compound: `(1 + r).cumprod()`, not `r.cumsum()`.
> Adding returns is Lecture 1's pitfall 4 and the error grows with horizon.

In [ ]:
# your code here


**What happened:**

---

## 📤 Submission

1. **Run your notebook from a clean start** — Runtime → Restart and run all.
   If it does not survive that, Q1 is still ahead of you.
2. File → Download → `.ipynb`
3. Upload to Brightspace under Assignment 1.

---

## What this was for

Next week you meet a dataset with 1.5 million rows where you cannot open the
file and look. Every trap in Part 3 is one you will hit again there, invisibly:
columns that are not what they claim, sentinel values masquerading as data,
dates that do not line up, and percent where you assumed decimal.

You will also notice that this file is **wide** — one column per industry. The
course panel is **long** — one row per stock-month. Lecture 2 explains why, and
having built something in the wide shape first is the point.